# Module 02 — Notebook 02: IG on Text Classification

## Learning Objectives
By completing this notebook, you will:
1. Apply LayerIntegratedGradients to a pretrained DistilBERT sentiment classifier
2. Convert token embedding attributions to per-token importance scores
3. Visualize token attributions as colored text (green=positive, red=negative)
4. Analyze which words most influence sentiment predictions

## Prerequisites
- Module 02 Notebook 01 completed
- Guide 02 (baselines) — specifically the section on text baselines

## Estimated Time: 14 minutes

---

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from transformers import AutoModelForSequenceClassification, AutoTokenizer
from captum.attr import LayerIntegratedGradients

torch.manual_seed(42)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {DEVICE}")

# Load pretrained DistilBERT for sentiment classification
# Fine-tuned on SST-2 (Stanford Sentiment Treebank — movie reviews)
MODEL_NAME = "distilbert-base-uncased-finetuned-sst-2-english"
print(f"Loading {MODEL_NAME}...")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME)
model = model.to(DEVICE)
model.eval()

LABEL_MAP = model.config.id2label
print(f"Model loaded. Labels: {LABEL_MAP}")

## 1. Why LayerIntegratedGradients for Text?

Standard IG attributes to *input* features. For text models, the input is token IDs (integers like `[101, 2023, 2003, ...]`). You cannot compute gradients with respect to integers.

**Solution:** Attribute to the *embedding layer* output. The embedding layer converts token IDs to continuous vectors. `LayerIntegratedGradients` computes IG at the embedding layer, giving us a per-token, per-dimension attribution. We then sum across the embedding dimension to get a single importance score per token.

In [ ]:
# LayerIntegratedGradients targets the embedding layer
# This is the layer that converts token IDs → continuous vectors
embedding_layer = model.distilbert.embeddings

lig = LayerIntegratedGradients(
    model,            # The full model (forward function)
    embedding_layer   # The layer to attribute TO
)

print(f"Target layer: {embedding_layer.__class__.__name__}")
print(f"Embedding dim: {model.config.hidden_size}")
print(f"Vocab size:    {model.config.vocab_size}")

# Inspect: what does the embedding layer output look like?
sample_ids = tokenizer("great movie", return_tensors="pt")["input_ids"]
with torch.no_grad():
    embed_out = embedding_layer(sample_ids)
print(f"\nEmbedding output shape: {embed_out.shape}")
print(f"  (batch=1, seq_len={embed_out.shape[1]}, embed_dim={embed_out.shape[2]})")
print("After LayerIG: attribution shape = (1, seq_len, embed_dim)")
print("Sum across embed_dim → (1, seq_len) = one score per token")

## 2. Token Attribution Function

We wrap the attribution computation in a reusable function that:
1. Tokenizes the input text
2. Computes LayerIG attributions
3. Sums across embedding dimension → per-token scores
4. Returns tokens and scores

In [ ]:
def get_token_attributions(text, target_class=None, n_steps=50):
    """
    Compute per-token IG attributions for a text input.

    Parameters
    ----------
    text : str
        Input text to explain
    target_class : int or None
        Target class index. If None, uses top predicted class.
    n_steps : int
        Number of IG integration steps

    Returns
    -------
    tokens : list of str
    token_attributions : ndarray (seq_len,)
    prediction_label : str
    prediction_prob : float
    convergence_delta : float
    """
    # Tokenize
    inputs = tokenizer(text, return_tensors="pt")
    input_ids     = inputs["input_ids"].to(DEVICE)
    attention_mask = inputs["attention_mask"].to(DEVICE)

    # Get prediction
    with torch.no_grad():
        outputs = model(**inputs.to(DEVICE))
        probs = torch.softmax(outputs.logits, dim=-1)
        pred_class = probs.argmax().item()
        pred_prob  = probs.max().item()

    if target_class is None:
        target_class = pred_class

    # Baseline: zero token IDs (same shape as input)
    # Using zero IDs means the baseline is the embedding of [PAD]
    baseline_ids = torch.zeros_like(input_ids)

    # LayerIG requires input_ids, but the layer is the embedding layer
    # We pass attention_mask as additional_forward_args
    attributions, delta = lig.attribute(
        inputs=input_ids,
        baselines=baseline_ids,
        target=target_class,
        additional_forward_args=(attention_mask,),
        n_steps=n_steps,
        return_convergence_delta=True
    )

    # Sum across embedding dimension → per-token score
    # Shape: (1, seq_len, embed_dim) → (seq_len,)
    token_attr = attributions.sum(dim=-1).squeeze(0).detach().cpu().numpy()

    # Get readable token strings
    tokens = tokenizer.convert_ids_to_tokens(input_ids[0].tolist())

    return (
        tokens,
        token_attr,
        LABEL_MAP[pred_class],
        pred_prob,
        delta.item()
    )


# Quick test
tokens, attrs, pred_label, pred_prob, delta = get_token_attributions(
    "This movie was absolutely fantastic!"
)
print(f"Tokens: {tokens}")
print(f"Attribution shape: {attrs.shape}")
print(f"Prediction: {pred_label} ({pred_prob:.1%})")
print(f"Convergence delta: {delta:.5f}")

## 3. Visualization: Colored Token Attribution

We visualize token attributions using color-coded text:
- **Green**: token supports positive sentiment prediction
- **Red**: token opposes positive sentiment (supports negative)
- **White/neutral**: token has little effect on prediction

In [ ]:
def visualize_token_attributions(tokens, attributions, title="",
                                  prediction_label="", prediction_prob=0.0,
                                  convergence_delta=0.0):
    """
    Create a colored-text visualization of token attributions.

    Green = supports positive sentiment
    Red   = opposes positive sentiment
    Intensity = attribution magnitude
    """
    # Normalize attributions to [-1, 1]
    max_abs = np.abs(attributions).max() + 1e-8
    attr_norm = attributions / max_abs

    # Filter: remove [CLS] and [SEP] tokens for cleaner display
    # (they are special tokens added by the tokenizer)
    display_tokens = []
    display_attrs = []
    for tok, att in zip(tokens, attr_norm):
        if tok not in ['[CLS]', '[SEP]', '[PAD]']:
            display_tokens.append(tok)
            display_attrs.append(att)

    n_tokens = len(display_tokens)
    fig_width = max(10, n_tokens * 0.95)

    fig, (ax_text, ax_bar) = plt.subplots(
        2, 1, figsize=(fig_width, 3.5),
        gridspec_kw={'height_ratios': [2, 1]}
    )

    # --- Text visualization ---
    ax_text.set_xlim(0, n_tokens + 0.5)
    ax_text.set_ylim(0, 1)
    ax_text.axis('off')

    x = 0.3
    for token, score in zip(display_tokens, display_attrs):
        intensity = min(abs(score) * 1.2 + 0.1, 0.95)
        if score > 0:
            facecolor = (0.0, intensity * 0.9, 0.0, 0.85)
        else:
            facecolor = (intensity * 0.9, 0.0, 0.0, 0.85)

        # Clean up subword tokens (remove ## prefix)
        display_tok = token.replace('##', '') if token.startswith('##') else token

        ax_text.text(
            x, 0.5, display_tok,
            ha='center', va='center',
            fontsize=12, fontweight='bold', color='white',
            bbox=dict(boxstyle='round,pad=0.3', facecolor=facecolor,
                      edgecolor='none', alpha=0.9)
        )
        x += max(len(display_tok) * 0.22 + 0.4, 0.8)

    ax_text.set_title(
        f"{title}\nPrediction: {prediction_label} ({prediction_prob:.1%}) | "
        f"Convergence δ = {convergence_delta:.4f}",
        fontsize=11
    )

    # --- Bar chart ---
    colors = ['#2ecc71' if v > 0 else '#e74c3c' for v in display_attrs]
    ax_bar.bar(range(n_tokens), display_attrs, color=colors, alpha=0.8, edgecolor='none')
    ax_bar.set_xticks(range(n_tokens))
    ax_bar.set_xticklabels(
        [t.replace('##', '') for t in display_tokens],
        rotation=30, ha='right', fontsize=8
    )
    ax_bar.axhline(0, color='black', linewidth=0.8)
    ax_bar.set_ylabel('Attribution Score', fontsize=9)

    # Legend
    legend_patches = [
        mpatches.Patch(color='#2ecc71', label='Positive (supports POSITIVE)'),
        mpatches.Patch(color='#e74c3c', label='Negative (supports NEGATIVE)')
    ]
    ax_bar.legend(handles=legend_patches, fontsize=8, loc='upper right')

    plt.tight_layout()
    plt.show()


# Demonstrate with a positive sentence
text_pos = "This movie was absolutely fantastic and I loved every moment of it!"
tokens_p, attrs_p, label_p, prob_p, delta_p = get_token_attributions(text_pos)
visualize_token_attributions(
    tokens_p, attrs_p,
    title=f'Text: "{text_pos}"',
    prediction_label=label_p, prediction_prob=prob_p,
    convergence_delta=delta_p
)

In [ ]:
# Negative sentiment example
text_neg = "Terrible acting and a boring predictable plot. Complete waste of time."
tokens_n, attrs_n, label_n, prob_n, delta_n = get_token_attributions(text_neg)
visualize_token_attributions(
    tokens_n, attrs_n,
    title=f'Text: "{text_neg}"',
    prediction_label=label_n, prediction_prob=prob_n,
    convergence_delta=delta_n
)

# Ambiguous sentence — what does the model do?
text_ambig = "Some parts were great but the ending was disappointing and left me confused."
tokens_a, attrs_a, label_a, prob_a, delta_a = get_token_attributions(text_ambig)
visualize_token_attributions(
    tokens_a, attrs_a,
    title=f'Text: "{text_ambig}"',
    prediction_label=label_a, prediction_prob=prob_a,
    convergence_delta=delta_a
)

## 4. Attribute for BOTH Classes

For the ambiguous sentence, we compute attribution for both the POSITIVE and NEGATIVE classes. This shows which words push towards positive vs negative independently.

In [ ]:
# Attribute for both classes on the ambiguous sentence
text = "Some parts were great but the ending was disappointing and left me confused."

tokens_0, attrs_0, _, prob_0, delta_0 = get_token_attributions(text, target_class=0)  # NEGATIVE
tokens_1, attrs_1, _, prob_1, delta_1 = get_token_attributions(text, target_class=1)  # POSITIVE

print(f"Ambiguous sentence: '{text}'")

# Compute model prediction
inputs = tokenizer(text, return_tensors="pt")
with torch.no_grad():
    probs = torch.softmax(model(**inputs.to(DEVICE)).logits, dim=-1)
print(f"NEGATIVE probability: {probs[0, 0]:.3f}")
print(f"POSITIVE probability: {probs[0, 1]:.3f}")

# Get display tokens (strip [CLS], [SEP])
all_tokens = tokenizer.convert_ids_to_tokens(inputs['input_ids'][0].tolist())
display_mask = [t not in ['[CLS]', '[SEP]', '[PAD]'] for t in all_tokens]
display_tokens = [t for t, m in zip(all_tokens, display_mask) if m]
attrs_0_disp = np.array([a for a, m in zip(attrs_0, display_mask) if m])
attrs_1_disp = np.array([a for a, m in zip(attrs_1, display_mask) if m])

# Side-by-side bar chart
n = len(display_tokens)
x = np.arange(n)
w = 0.4

# Normalize for comparison
max_abs = max(np.abs(attrs_0_disp).max(), np.abs(attrs_1_disp).max()) + 1e-8
attrs_0_norm = attrs_0_disp / max_abs
attrs_1_norm = attrs_1_disp / max_abs

fig, ax = plt.subplots(figsize=(max(10, n * 0.85), 5))
ax.bar(x - w/2, attrs_0_norm, w, label='Attribution for NEGATIVE class',
        color='#e74c3c', alpha=0.8)
ax.bar(x + w/2, attrs_1_norm, w, label='Attribution for POSITIVE class',
        color='#2ecc71', alpha=0.8)
ax.set_xticks(x)
ax.set_xticklabels([t.replace('##', '') for t in display_tokens],
                    rotation=30, ha='right', fontsize=9)
ax.axhline(0, color='black', lw=0.8)
ax.set_ylabel('Normalized Attribution')
ax.set_title(
    f'Attribution for BOTH classes — Ambiguous sentence\n'
    f'Model prediction: NEGATIVE ({probs[0,0]:.1%}) / POSITIVE ({probs[0,1]:.1%})',
    fontsize=11
)
ax.legend()
plt.tight_layout()
plt.show()

print("\nObservations:")
print("'great', 'parts': positive attribution for POSITIVE class")
print("'disappointing', 'confused': positive attribution for NEGATIVE class")
print("The model sees conflicting signals — reflected in lower confidence.")

## Summary

### What You Learned

1. **LayerIntegratedGradients** attributes to intermediate layer outputs — essential for text models where inputs are non-differentiable token IDs
2. **PAD token baseline** represents "no token information" in transformer models
3. **Sum across embedding dim** converts per-dimension attributions to per-token scores
4. **Signed attribution** distinguishes words that support vs. oppose each sentiment class
5. **Dual-class attribution** on ambiguous text reveals the conflicting signals the model balances

### What's Next

**Notebook 03:** NoiseTunnel SmoothGrad on IG — variance reduction demo comparing plain IG, SmoothGrad-IG, and VarGrad-IG.